# Интерпретация

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'


### Загрузка и подготовка данных


Вам будет предоставлен датасет, посвященный продаже недвижимости. Ваша задача - построить интерпретацию для этого датасета. В нем достаточно много различных признаков, поэтому вы можете предварительно отфильтровать их, когда будете строить графики. Оставляйте достаточно признаков, чтобы ваши модели оставались точными..

In [ ]:
data_path = r'C:\Users\gosha\Downloads\Telegram Desktop\hw_interpretation\hw_interpretation\data.csv'
data = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {data.shape}")
print(f"\nПервые строки:")
data.head()


Размер датасета: (93455, 83)

Первые строки:


,region_name_cat,district_cat,corpus_cat,developer_cat,agreement_date,floor,square,rooms_4,location_logs_count_mean,location_depth,...,location_public_transport_platform_w_mean_distance,location_water_w_mean_distance,location_university_w_mean_distance,location_leisure_w_mean_distance,location_pop_shop_cnt,price_target,hc_name_cat,interior_cat,class_cat,stage_cat
0,Город,58,1331,91,2012-04-13,10.0,78.44,3,23.131066,13.0,...,0.894947,0.772872,1.309514,0.853183,7.0,25024.299281,36,0.0,97865,27728
1,Город,75,1677,91,2013-09-16,2.0,34.15,1,14.090185,13.0,...,1.063211,0.840130,-999.000000,1.147596,0.0,18477.300271,372,32413.0,97865,70661
2,Пригород,48,316,10,2014-07-31,17.0,59.85,2,19.453795,13.0,...,0.832622,-999.000000,-999.000000,0.905435,1.0,17441.013879,336,8977.0,97865,12638
3,Пригород,48,1409,91,2012-12-30,12.0,67.53,2,13.178136,13.0,...,-999.000000,1.322121,-999.000000,1.263878,0.0,17019.139763,154,32413.0,97865,70661
4,Пригород,48,1590,91,2014-06-20,5.0,58.13,2,20.691770,13.0,...,0.825295,0.996629,-999.000000,1.024595,2.0,17132.394908,154,32413.0,97865,12638


## Задание 1. 1 балл
Сделайте 2 версии данных - с нормализацией признаков и без.
Обучите 6 моделей:
- линейную регрессию (LinearRegression) на двух вариантах данных
- Lasso регрессию (Lasso) на двух вариантах данных
- градиентный бустинг (GradientBoostingRegressor) на двух вариантах данных. Ограничьте глубину до 5.

Выведите MSE,RMSE и MAPE моделей. Какая функция больше подходит? Почему?

Зафиксируйте выводы. Какие модели чувствительны к масштабу признаков, а какие почти инвариантны? Почему это важно для анализа признаков?

In [ ]:
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

target_col = "price_target"

model_data = data.copy()
model_data["agreement_date"] = pd.to_datetime(model_data["agreement_date"])
model_data["agreement_year"] = model_data["agreement_date"].dt.year
model_data["agreement_month"] = model_data["agreement_date"].dt.month
model_data["agreement_dayofyear"] = model_data["agreement_date"].dt.dayofyear
model_data = model_data.drop(columns=["agreement_date"])

all_categorical_cols = [col for col in model_data.columns if col.endswith("_cat")] + ["rooms_4"]
all_categorical_cols = [col for col in all_categorical_cols if col in model_data.columns and col != target_col]
model_data[all_categorical_cols] = model_data[all_categorical_cols].astype("string").fillna("missing")
all_numeric_cols = [col for col in model_data.columns if col not in all_categorical_cols + [target_col]]

num_rank = (
    model_data[all_numeric_cols + [target_col]]
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .abs()
    .sort_values(ascending=False)
)
selected_numeric_cols = num_rank.head(45).index.tolist()
selected_categorical_cols = ["region_name_cat", "rooms_4", "district_cat", "class_cat", "stage_cat"]
selected_features = selected_numeric_cols + selected_categorical_cols

X = model_data[selected_features]
y = model_data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)


def make_preprocessor(scaler=None):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scaler is not None:
        numeric_steps.append(("scaler", scaler))

    categorical_steps = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        [
            ("num", Pipeline(numeric_steps), selected_numeric_cols),
            ("cat", categorical_steps, selected_categorical_cols),
        ],
        remainder="drop",
    )


preprocessors = {
    "raw": make_preprocessor(),
    "minmax": make_preprocessor(MinMaxScaler()),
}
X_train_versions = {}
X_test_versions = {}
feature_names = {}

for version, preprocessor in preprocessors.items():
    X_train_versions[version] = preprocessor.fit_transform(X_train)
    X_test_versions[version] = preprocessor.transform(X_test)
    feature_names[version] = preprocessor.get_feature_names_out()

base_estimators = {
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(alpha=0.5, max_iter=5000, random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=42,
        n_estimators=80,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
    ),
}

models = {name: {} for name in base_estimators}
metrics_rows = []

for model_name, estimator in base_estimators.items():
    for version in ["raw", "minmax"]:
        model = clone(estimator)
        model.fit(X_train_versions[version], y_train)
        predictions = model.predict(X_test_versions[version])
        models[model_name][version] = model
        metrics_rows.append(
            {
                "preprocessing": version,
                "model": model_name,
                "MSE": mean_squared_error(y_test, predictions),
                "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
                "MAPE": mean_absolute_percentage_error(y_test, predictions),
                "R2": r2_score(y_test, predictions),
            }
        )

metrics_df = pd.DataFrame(metrics_rows).sort_values(["preprocessing", "R2"], ascending=[True, False])

display(metrics_df.round(4))
print(f"Selected features: {len(selected_numeric_cols)} numeric + {len(selected_categorical_cols)} categorical")
print("MAPE is the easiest metric to interpret here because the target is price and the error becomes relative.")
print("LinearRegression is almost prediction-invariant to scaling, Lasso is more sensitive because of the L1 penalty, and GradientBoostingRegressor is nearly scale-invariant.")

analysis_feature = "square"
analysis_version = "minmax"
baseline_value = X_train[analysis_feature].median()


def clean_feature_names(names):
    cleaned = []
    for name in names:
        name = name.replace("num__", "").replace("cat__", "")
        name = name.replace("region_name_cat_", "region=")
        name = name.replace("rooms_4_", "rooms=")
        name = name.replace("district_cat_", "district=")
        name = name.replace("class_cat_", "class=")
        name = name.replace("stage_cat_", "stage=")
        cleaned.append(name)
    return cleaned


class RawModelWrapper:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def predict(self, X_raw):
        return self.model.predict(self.preprocessor.transform(X_raw))


wrapped_models = {
    model_name: RawModelWrapper(preprocessors[analysis_version], models[model_name][analysis_version])
    for model_name in ["LinearRegression", "GradientBoostingRegressor"]
}


def get_importance(model_name, version):
    model = models[model_name][version]
    if model_name in {"LinearRegression", "Lasso"}:
        importance = np.abs(model.coef_)
    else:
        importance = model.feature_importances_
    return pd.Series(importance, index=feature_names[version]).sort_values(ascending=False)


def predict_raw(model_name, version, X_raw):
    return models[model_name][version].predict(preprocessors[version].transform(X_raw))


def compute_effect_curves(model_name, version, X_raw, feature, grid_values):
    curves = []
    for value in grid_values:
        X_modified = X_raw.copy()
        X_modified.loc[:, feature] = value
        curves.append(predict_raw(model_name, version, X_modified))
    return np.vstack(curves).T


## Задание 1.1(*) 1 балл
Сравните модели, построенные с помощью разных видов нормализации (MinMax, Standart). Отличается ли важность признаков?

In [ ]:
from sklearn.preprocessing import StandardScaler

if "standard" not in preprocessors:
    preprocessors["standard"] = make_preprocessor(StandardScaler())
    X_train_versions["standard"] = preprocessors["standard"].fit_transform(X_train)
    X_test_versions["standard"] = preprocessors["standard"].transform(X_test)
    feature_names["standard"] = preprocessors["standard"].get_feature_names_out()

    for model_name, estimator in base_estimators.items():
        standard_model = clone(estimator)
        standard_model.fit(X_train_versions["standard"], y_train)
        models[model_name]["standard"] = standard_model

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, model_name in zip(axes, ["LinearRegression", "Lasso", "GradientBoostingRegressor"]):
    minmax_importance = get_importance(model_name, "minmax")
    standard_importance = get_importance(model_name, "standard")

    top_features = minmax_importance.head(8).index.union(standard_importance.head(8).index)
    comparison = pd.DataFrame(
        {
            "MinMax": minmax_importance.reindex(top_features, fill_value=0.0),
            "Standard": standard_importance.reindex(top_features, fill_value=0.0),
        }
    )
    comparison["avg"] = comparison.mean(axis=1)
    comparison = comparison.sort_values("avg").drop(columns="avg")

    positions = np.arange(len(comparison))
    ax.barh(positions - 0.2, comparison["MinMax"], height=0.4, label="MinMax")
    ax.barh(positions + 0.2, comparison["Standard"], height=0.4, label="Standard")
    ax.set_yticks(positions)
    ax.set_yticklabels(clean_feature_names(comparison.index))
    ax.set_title(model_name)
    ax.set_xlabel("importance")
    ax.legend(loc="best")

plt.tight_layout()

for model_name in ["LinearRegression", "Lasso", "GradientBoostingRegressor"]:
    minmax_top10 = set(get_importance(model_name, "minmax").head(10).index)
    standard_top10 = set(get_importance(model_name, "standard").head(10).index)
    print(f"{model_name}: top-10 overlap = {len(minmax_top10 & standard_top10)}/10")

print("Coefficient-based importance changes after rescaling for LinearRegression and Lasso.")
print("GradientBoostingRegressor stays almost unchanged because tree splits depend on order, not on measurement units.")


## Задание 2. 1 балл
Выберите 1 признак для анализа (можно категориальный, с не менее чем 5 уровнями, или дискретизируйте непрерывный). 
Используйте линейную регрессию и бустинг после применения MinMaxScaler. Что будет с моделями, если признаки выйдут из диапазона?
Постройте графики ICE и PDP для интерпретации исходных данных, а также искусственно добавив несколько выбросов, выходящих за оригинальные интервалы. 

Задание 2.1 (*) 1 балл: проанализируйте также еще один признак

## Задание 3. 1 балл
Выберите 20 объектов из тестовой выборки.
Для каждого объекта из выбранного набора построим траекторию изменения предсказания модели при постепенном изменении значения признака от его текущего значения к базовому значению (медиана или среднее по обучающей выборке).

**Алгоритм:**
1. Выбрать объект $x_i$ из тестовой выборки
2. Для интересующего признака $j$:
   - Текущее значение: $x_{i,j}$
   - Базовое значение: $x_{base,j}$ (медиана или среднее по обучающей выборке)
3. Построить линейную интерполяцию между $x_{i,j}$ и $x_{base,j}$ с $n$ шагами
4. Для каждого шага интерполяции:
   - Заменить значение признака $j$ в объекте $x_i$ на значение из интерполяции
   - Вычислить предсказание модели для модифицированного объекта
5. Построить график траектории


Задание 3.1 (*) 1 балл: проанализируйте также еще один признак

In [ ]:
print(f"Feature for analysis: {analysis_feature}")
print(
    f"Train range: [{X_train[analysis_feature].min():.2f}, {X_train[analysis_feature].max():.2f}], "
    f"train median: {baseline_value:.2f}"
)

sample_original = X_test.sample(n=150, random_state=42).copy()
outliers_sample = sample_original.copy()
outliers_sample.iloc[:5, outliers_sample.columns.get_loc(analysis_feature)] = X_train[analysis_feature].max() * 1.5
outliers_sample.iloc[5:10, outliers_sample.columns.get_loc(analysis_feature)] = max(1.0, X_train[analysis_feature].min() * 0.5)

original_grid = np.linspace(
    sample_original[analysis_feature].quantile(0.05),
    sample_original[analysis_feature].quantile(0.95),
    30,
)
outlier_grid = np.linspace(
    outliers_sample[analysis_feature].min(),
    outliers_sample[analysis_feature].max(),
    30,
)


def plot_ice_pdp(ax, model_name, X_raw, grid_values, title):
    curves = compute_effect_curves(model_name, analysis_version, X_raw, analysis_feature, grid_values)
    subset_idx = np.linspace(0, len(curves) - 1, min(25, len(curves)), dtype=int)

    for curve in curves[subset_idx]:
        ax.plot(grid_values, curve, color="steelblue", alpha=0.15)

    ax.plot(grid_values, curves.mean(axis=0), color="darkred", linewidth=3, label="PDP")
    ax.axvline(X_train[analysis_feature].min(), color="gray", linestyle="--", linewidth=1)
    ax.axvline(X_train[analysis_feature].max(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(analysis_feature)
    ax.legend(loc="best")


fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
plot_ice_pdp(axes[0, 0], "LinearRegression", sample_original, original_grid, "LinearRegression: original range")
plot_ice_pdp(axes[0, 1], "GradientBoostingRegressor", sample_original, original_grid, "GBR: original range")
plot_ice_pdp(axes[1, 0], "LinearRegression", outliers_sample, outlier_grid, "LinearRegression: with outliers")
plot_ice_pdp(axes[1, 1], "GradientBoostingRegressor", outliers_sample, outlier_grid, "GBR: with outliers")
axes[0, 0].set_ylabel("prediction")
axes[1, 0].set_ylabel("prediction")
plt.tight_layout()

selected_objects = X_test.sample(n=20, random_state=7)
trajectory_grid = np.linspace(0, 1, 35)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, model_name in zip(axes, ["LinearRegression", "GradientBoostingRegressor"]):
    for _, row in selected_objects.iterrows():
        path = pd.DataFrame([row] * len(trajectory_grid))
        start_value = row[analysis_feature]
        path_values = start_value + (baseline_value - start_value) * trajectory_grid
        path[analysis_feature] = path_values
        predictions = predict_raw(model_name, analysis_version, path)
        ax.plot(path_values, predictions, alpha=0.55, linewidth=1.1)

    ax.axvline(baseline_value, color="black", linestyle="--", linewidth=2, label="train median")
    ax.set_title(f"Trajectories: {model_name}")
    ax.set_xlabel(analysis_feature)
    ax.legend(loc="best")

axes[0].set_ylabel("prediction")
plt.tight_layout()

print("Outside the MinMax train range, LinearRegression keeps extrapolating linearly.")
print("GradientBoostingRegressor mostly saturates near the border leaves, so the PDP becomes flatter outside the seen range.")


## Задание 4 (1 балл). ALE
Постройте ALE по обеим моделям, используя pyALE. Подберите размер сетки так, чтобы получить доверительные интервалы. Проанализируйте полученный график. Каковы получились доверительные интервалы? Почему они различны для моделей?

P.s. Сетку значений стройте для исходного признака.

In [ ]:
from PyALE import ale

ale_sample = X_train.sample(n=5000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
ale_results = {}

for ax, model_name in zip(axes, ["LinearRegression", "GradientBoostingRegressor"]):
    ale_results[model_name] = ale(
        X=ale_sample,
        model=wrapped_models[model_name],
        feature=[analysis_feature],
        grid_size=20,
        include_CI=True,
        plot=True,
        fig=fig,
        ax=ax,
    )
    ax.set_title(f"ALE: {model_name}")

plt.tight_layout()

for model_name, result in ale_results.items():
    ci_width = (result["upperCI_95%"] - result["lowerCI_95%"]).dropna()
    print(
        f"{model_name}: mean CI width = {ci_width.mean():.2f}, "
        f"max CI width = {ci_width.max():.2f}"
    )

print("A grid size of 20 keeps enough points inside bins and still gives visible confidence intervals.")
print("For LinearRegression the local effect is nearly constant, so the intervals are almost zero.")
print("For GradientBoostingRegressor the effect is piecewise and depends on local data density, so the intervals are wider and uneven.")


## Задание 5:  Permutation Importance (2 балла)
Постройте Permutation importances по обеим моделям, используя sklearn.

Поэкспериментируйте с числом перестановок.

Проанализируйте полученные коэффициенты. Как они меняются от количества перестановок? Как меняются std коэффициентов?



In [ ]:
from sklearn.inspection import permutation_importance

perm_rng = np.random.default_rng(42)
perm_idx = perm_rng.choice(X_test_versions[analysis_version].shape[0], size=750, replace=False)
X_perm = X_test_versions[analysis_version][perm_idx]
y_perm = y_test.iloc[perm_idx]
repeats_list = [3, 10, 20]
perm_results = {model_name: {} for model_name in ["LinearRegression", "GradientBoostingRegressor"]}

for model_name in perm_results:
    for repeats in repeats_list:
        perm_results[model_name][repeats] = permutation_importance(
            models[model_name][analysis_version],
            X_perm,
            y_perm,
            n_repeats=repeats,
            random_state=42,
            scoring="r2",
            n_jobs=1,
        )

fig, axes = plt.subplots(len(repeats_list), 2, figsize=(16, 14))

for row, repeats in enumerate(repeats_list):
    for col, model_name in enumerate(["LinearRegression", "GradientBoostingRegressor"]):
        result = perm_results[model_name][repeats]
        mean_importance = pd.Series(result.importances_mean, index=feature_names[analysis_version])
        std_importance = pd.Series(result.importances_std, index=feature_names[analysis_version])
        top_features = mean_importance.sort_values(ascending=False).head(8).sort_values()

        axes[row, col].barh(
            clean_feature_names(top_features.index),
            top_features.values,
            xerr=std_importance.reindex(top_features.index).values,
            color="teal" if model_name == "LinearRegression" else "darkorange",
            alpha=0.85,
        )
        axes[row, col].set_title(f"{model_name}, n_repeats={repeats}")
        axes[row, col].set_xlabel("drop in R2")

plt.tight_layout()

for model_name in ["LinearRegression", "GradientBoostingRegressor"]:
    print(model_name)
    for repeats in repeats_list:
        result = perm_results[model_name][repeats]
        top3 = pd.Series(result.importances_mean, index=feature_names[analysis_version]).sort_values(ascending=False).head(3)
        summary = ", ".join(f"{name}={value:.3f}" for name, value in zip(clean_feature_names(top3.index), top3.values))
        print(f"  n_repeats={repeats}: {summary}")

print("As n_repeats grows, the ranking becomes more stable and the error bars are less noisy.")


## Задание 5: Feature Importance (2 балла)
Пусть важность - это MAPE для тестовых данных. Проведите анализ только для бустинга

Идея перестановочной важности представляет собой частный случай важности при помощи внесения возмущений в признак. Примеры возмущений:
1) внесение случайного шума
2) зануление признака
3) сдвиг признака к его базовому значению и оценка траектории изменения прогнозов или качества модели 

Примем за базовое значение (${base}$)медиану признака и будем сдвигать исходный признак к медианному с некоторым коэффициентом $\beta$:
$x_j^\beta = (1- \beta)x_j + \beta {base}$

Реализуйте это возмущение. Как меняются важности при разных $\beta$?

Постройте графики важности и сравните важности с permutation importance. Используйте только числовые признаки. При этом медиану стоит считать на тренировочном наборе, а важность как разницу MAPE на тестовой выборке. Чем больше разница, тем важнее признак. 


Сравните результаты методов. Какие признаки наиболее важны? Есть ли различия между методами? В чём могут быть причины различий?

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

gbr_model = models["GradientBoostingRegressor"][analysis_version]
base_predictions = gbr_model.predict(X_test_versions[analysis_version])
base_mape = mean_absolute_percentage_error(y_test, base_predictions)
beta_grid = [0.25, 0.5, 0.75, 1.0]
perturb_rng = np.random.default_rng(42)
feature_importance_rows = []

for feature in selected_numeric_cols:
    x_noise = X_test.copy()
    noise_scale = X_train[feature].std()
    x_noise.loc[:, feature] = x_noise[feature] + perturb_rng.normal(0, noise_scale, size=len(x_noise))
    noise_mape = mean_absolute_percentage_error(
        y_test,
        predict_raw("GradientBoostingRegressor", analysis_version, x_noise),
    )
    feature_importance_rows.append(
        {"feature": feature, "method": "noise", "beta": np.nan, "delta_mape": noise_mape - base_mape}
    )

    x_zero = X_test.copy()
    x_zero.loc[:, feature] = 0.0
    zero_mape = mean_absolute_percentage_error(
        y_test,
        predict_raw("GradientBoostingRegressor", analysis_version, x_zero),
    )
    feature_importance_rows.append(
        {"feature": feature, "method": "zero", "beta": np.nan, "delta_mape": zero_mape - base_mape}
    )

    base_value = X_train[feature].median()
    for beta in beta_grid:
        x_shift = X_test.copy()
        x_shift.loc[:, feature] = (1 - beta) * x_shift[feature] + beta * base_value
        shift_mape = mean_absolute_percentage_error(
            y_test,
            predict_raw("GradientBoostingRegressor", analysis_version, x_shift),
        )
        feature_importance_rows.append(
            {"feature": feature, "method": "shift", "beta": beta, "delta_mape": shift_mape - base_mape}
        )

feature_importance_df = pd.DataFrame(feature_importance_rows)
noise_importance = feature_importance_df.query("method == 'noise'").sort_values("delta_mape", ascending=False)
zero_importance = feature_importance_df.query("method == 'zero'").sort_values("delta_mape", ascending=False)
shift_importance = feature_importance_df.query("method == 'shift'")
full_shift_importance = shift_importance[shift_importance["beta"] == 1.0].sort_values("delta_mape", ascending=False)

perm_numeric = pd.Series(
    perm_results["GradientBoostingRegressor"][20].importances_mean,
    index=feature_names[analysis_version],
)
perm_numeric = perm_numeric[perm_numeric.index.str.startswith("num__")].sort_values(ascending=False)
perm_numeric.index = perm_numeric.index.str.replace("^num__", "", regex=True)


def plot_top_importance(ax, values, title, color):
    top_values = values.head(10).iloc[::-1]
    ax.barh(top_values.index, top_values.values, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("delta MAPE" if "Permutation" not in title else "drop in R2")


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plot_top_importance(axes[0, 0], noise_importance.set_index("feature")["delta_mape"], "Noise perturbation", "steelblue")
plot_top_importance(axes[0, 1], zero_importance.set_index("feature")["delta_mape"], "Zeroing", "seagreen")
plot_top_importance(axes[1, 0], full_shift_importance.set_index("feature")["delta_mape"], "Shift to train median (beta=1)", "darkorange")
plot_top_importance(axes[1, 1], perm_numeric, "Permutation importance (n_repeats=20)", "firebrick")
plt.tight_layout()

shift_top_features = full_shift_importance.head(6)["feature"]
shift_pivot = shift_importance.pivot(index="beta", columns="feature", values="delta_mape")
fig, ax = plt.subplots(figsize=(10, 6))
for feature in shift_top_features:
    ax.plot(shift_pivot.index, shift_pivot[feature], marker="o", linewidth=2, label=feature)
ax.set_title("How importance changes with beta")
ax.set_xlabel("beta")
ax.set_ylabel("delta MAPE")
ax.legend(loc="best")
plt.tight_layout()

comparison_table = pd.DataFrame(
    {
        "noise": noise_importance.set_index("feature")["delta_mape"],
        "zero": zero_importance.set_index("feature")["delta_mape"],
        "shift_beta_1": full_shift_importance.set_index("feature")["delta_mape"],
        "perm_r2_drop": perm_numeric,
    }
).fillna(0.0)
comparison_score = comparison_table.rank(ascending=False).mean(axis=1)
display(comparison_table.loc[comparison_score.sort_values().head(12).index].round(4))

print(f"Base GBR test MAPE: {base_mape:.4f}")
print("Noise top-3:", ", ".join(noise_importance.head(3)["feature"]))
print("Zero top-3:", ", ".join(zero_importance.head(3)["feature"]))
print("Shift top-3:", ", ".join(full_shift_importance.head(3)["feature"]))
print("Permutation top-3:", ", ".join(perm_numeric.head(3).index))
print("Importance grows with beta because the feature is moved farther from its original value toward the train median.")
print("The leaders overlap with permutation importance, but the ranking differs because each perturbation breaks the data manifold in its own way.")


#  Задание 6. 2 балла. LIME.
Постройте интерпретацию признаков для нескольких примеров с помощью LIME. Можете использовать 
Оцените устойчивость реализации. Как влияет на коэффициенты количество сгенерированных точек? А выбор признаков (lasso/добавление фичей по порядку). А выбор ядра?

(*) Вы получите на 2 балла больше, если используете свою реализацию из задания семинарского ноутбука. В таком случае не забудьте добавить тесты для своей реализации. 



In [ ]:
import lime.lime_tabular

lime_background_idx = np.random.default_rng(42).choice(
    X_train_versions[analysis_version].shape[0],
    size=2500,
    replace=False,
)
lime_feature_names = clean_feature_names(feature_names[analysis_version])


def make_lime_explainer(feature_selection="auto", kernel_width=None, kernel=None):
    return lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_versions[analysis_version][lime_background_idx],
        feature_names=lime_feature_names,
        mode="regression",
        discretize_continuous=False,
        random_state=42,
        kernel_width=kernel_width,
        kernel=kernel,
        feature_selection=feature_selection,
    )


def top_abs_lime_weights(explanation, top_k=8):
    weights = pd.DataFrame(explanation.as_list(), columns=["feature", "weight"])
    weights["abs_weight"] = weights["weight"].abs()
    return weights.sort_values("abs_weight", ascending=False).head(top_k).sort_values("weight")


def lime_weights(
    model_name,
    row_position,
    num_samples=2000,
    feature_selection="auto",
    kernel_width=None,
    kernel=None,
    num_features=8,
):
    explainer = make_lime_explainer(
        feature_selection=feature_selection,
        kernel_width=kernel_width,
        kernel=kernel,
    )
    explanation = explainer.explain_instance(
        X_test_versions[analysis_version][row_position],
        models[model_name][analysis_version].predict,
        num_features=num_features,
        num_samples=num_samples,
    )
    return top_abs_lime_weights(explanation, top_k=num_features)


def plot_lime_weights(ax, weights, title):
    colors = np.where(weights["weight"] >= 0, "darkgreen", "firebrick")
    ax.barh(weights["feature"], weights["weight"], color=colors, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("local weight")


gbr_test_predictions = models["GradientBoostingRegressor"][analysis_version].predict(X_test_versions[analysis_version])
quantile_targets = np.quantile(gbr_test_predictions, [0.1, 0.5, 0.9])
example_positions = []
for target_value in quantile_targets:
    example_positions.append(int(np.argmin(np.abs(gbr_test_predictions - target_value))))
example_positions = list(dict.fromkeys(example_positions))

fig, axes = plt.subplots(2, len(example_positions), figsize=(18, 9))
for col, row_position in enumerate(example_positions):
    row_index = X_test.index[row_position]
    title_suffix = f"idx={row_index}, pred={gbr_test_predictions[row_position]:.0f}"
    for row, model_name in enumerate(["LinearRegression", "GradientBoostingRegressor"]):
        weights = lime_weights(model_name, row_position, num_samples=2000)
        plot_lime_weights(axes[row, col], weights, f"{model_name} | {title_suffix}")
plt.tight_layout()

reference_position = int(np.argmax(np.abs(y_test.to_numpy() - gbr_test_predictions)))
default_kernel_width = np.sqrt(X_train_versions[analysis_version].shape[1]) * 0.75
triangular_kernel = lambda d, kernel_width: np.maximum(0, 1 - d / kernel_width)
stability_configs = {
    "samples=1000": {"num_samples": 1000},
    "samples=4000": {"num_samples": 4000},
    "forward_selection": {"feature_selection": "forward_selection"},
    "lasso_path": {"feature_selection": "lasso_path"},
    "wide_kernel": {"kernel_width": default_kernel_width * 2},
    "triangular_kernel": {"kernel": triangular_kernel},
}

stability_frames = {}
for label, params in stability_configs.items():
    weights = lime_weights("GradientBoostingRegressor", reference_position, **params)
    stability_frames[label] = weights.set_index("feature")["weight"]

stability_matrix = pd.DataFrame(stability_frames).fillna(0.0)
top_stability_features = stability_matrix.abs().max(axis=1).sort_values(ascending=False).head(10).index
plt.figure(figsize=(11, 6))
sns.heatmap(
    stability_matrix.loc[top_stability_features],
    annot=True,
    fmt=".0f",
    cmap="coolwarm",
    center=0,
)
plt.title("LIME stability on one reference object")
plt.tight_layout()
display(stability_matrix.loc[top_stability_features].round(1))

print(f"Reference object for stability: test idx={X_test.index[reference_position]}")
print("More generated samples usually make the explanation smoother and reduce random fluctuations.")
print("Changing feature selection can noticeably change which features survive; lasso_path is the most aggressive setting here.")
print("Changing the kernel also changes weights, because locality is defined differently and distant synthetic points receive different importance.")


## Задание 7. 1 балл. SHAP
Постройте локальный график с SHAP для объекта с индексом, равным вашему номеру в таблице курса на обеих моделях и сделайте выводы. 
## Задание 7.1 (*). 1 балл.  Shap и категориальные переменные.
Shap разлагает предсказание модели вблизи точки x на базовый уровень и сумму вкладов признаков: $ f(x) = base + \sum_i{\phi_i(x)}$. В случае one-hot вклад признака - это сумма вкладов dummy столбцов. Сравните вклады категориальных признаков до и после кодировки - так ли это? 


In [ ]:
import shap
from sklearn.preprocessing import OrdinalEncoder

course_index = 2
assert course_index in X.index
target_object = X.loc[[course_index]].copy()
print(f"Course index used for SHAP: {course_index}")

background_idx = np.random.default_rng(42).choice(
    X_train_versions[analysis_version].shape[0],
    size=500,
    replace=False,
)
background = X_train_versions[analysis_version][background_idx]
feature_labels = clean_feature_names(feature_names[analysis_version])


def to_local_explanation(shap_output, data_row, names):
    values = np.asarray(shap_output.values).reshape(-1)
    base_values = np.asarray(shap_output.base_values).reshape(-1)[0]
    data_values = np.asarray(data_row).reshape(-1)
    return shap.Explanation(
        values=values,
        base_values=base_values,
        data=data_values,
        feature_names=names,
    )


linear_explainer = shap.LinearExplainer(models["LinearRegression"][analysis_version], background)
tree_explainer = shap.TreeExplainer(models["GradientBoostingRegressor"][analysis_version], data=background)
target_onehot = preprocessors[analysis_version].transform(target_object)
linear_local = to_local_explanation(
    linear_explainer(target_onehot),
    target_onehot[0],
    feature_labels,
)
tree_local = to_local_explanation(
    tree_explainer(target_onehot),
    target_onehot[0],
    feature_labels,
)

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
shap.plots.waterfall(linear_local, max_display=12, show=False)
plt.title("LinearRegression")
plt.subplot(1, 2, 2)
shap.plots.waterfall(tree_local, max_display=12, show=False)
plt.title("GradientBoostingRegressor")
plt.tight_layout()


ordinal_preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", MinMaxScaler()),
                ]
            ),
            selected_numeric_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                    ),
                ]
            ),
            selected_categorical_cols,
        ),
    ],
    remainder="drop",
)
X_train_ordinal = ordinal_preprocessor.fit_transform(X_train)
target_ordinal = ordinal_preprocessor.transform(target_object)
ordinal_feature_names = ordinal_preprocessor.get_feature_names_out().tolist()
ordinal_background = X_train_ordinal[background_idx]
ordinal_models = {
    "LinearRegression": LinearRegression().fit(X_train_ordinal, y_train),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=42,
        n_estimators=80,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
    ).fit(X_train_ordinal, y_train),
}
ordinal_linear_explainer = shap.LinearExplainer(ordinal_models["LinearRegression"], ordinal_background)
ordinal_tree_explainer = shap.TreeExplainer(ordinal_models["GradientBoostingRegressor"], data=ordinal_background)
ordinal_linear_local = to_local_explanation(
    ordinal_linear_explainer(target_ordinal),
    target_ordinal[0],
    clean_feature_names(ordinal_feature_names),
)
ordinal_tree_local = to_local_explanation(
    ordinal_tree_explainer(target_ordinal),
    target_ordinal[0],
    clean_feature_names(ordinal_feature_names),
)


def aggregate_onehot_categorical(local_explanation, raw_names):
    values = pd.Series(np.asarray(local_explanation.values).reshape(-1), index=raw_names)
    return pd.Series(
        {
            column: values[values.index.str.startswith(f"cat__{column}_")].sum()
            for column in selected_categorical_cols
        }
    )


def extract_ordinal_categorical(local_explanation, raw_names):
    values = pd.Series(np.asarray(local_explanation.values).reshape(-1), index=raw_names)
    return pd.Series({column: values.get(f"cat__{column}", 0.0) for column in selected_categorical_cols})


categorical_comparison = pd.DataFrame(
    {
        "onehot_linear_sum": aggregate_onehot_categorical(linear_local, feature_names[analysis_version]),
        "ordinal_linear": extract_ordinal_categorical(ordinal_linear_local, ordinal_feature_names),
        "onehot_gbr_sum": aggregate_onehot_categorical(tree_local, feature_names[analysis_version]),
        "ordinal_gbr": extract_ordinal_categorical(ordinal_tree_local, ordinal_feature_names),
    }
)
display(categorical_comparison.round(3))

print("Within the one-hot model, the categorical contribution is the sum of SHAP values over its dummy columns.")
print("After replacing one-hot by ordinal encoding, the contributions are not identical because the model itself changes together with the representation.")
